In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ethereum-transactions-for-fraud-detection/first_order_df.csv
/kaggle/input/ethereum-transactions-for-fraud-detection/second_order_df.csv


understanding the dataset. checking row names.

In [2]:
df = pd.read_csv("/kaggle/input/ethereum-transactions-for-fraud-detection/second_order_df.csv")
print(df.head())
print(df.info())


   Unnamed: 0                                             TxHash  BlockHeight  \
0           0  0x8ce1d46abe9e2be0712ecc078bc8f3234ba4a4568e33...      5659116   
1           1  0x8ae1819a3202c31f9305282e36bed4825a949813a775...      5679819   
2           2  0x1dd9fa9a81528d66791bd3f5e8ea0583aa126e897c6c...      5686368   
3           3  0x9e016e0b56518ef7e3b85590469dd44ead20eaebda48...      5696896   
4           4  0xaca3850ba0080cf47b47f80e46da452f61bcbb5470d3...      5848095   

    TimeStamp                                        From  \
0  1527017753  0xd551234ae421e3bcba99a0da6d736074f22192ff   
1  1527335634  0x002bf459dc58584d58886169ea0e80f3ca95ffaf   
2  1527435806  0xd551234ae421e3bcba99a0da6d736074f22192ff   
3  1527595105  0x002bf459dc58584d58886169ea0e80f3ca95ffaf   
4  1529873859  0x16f209b5332a1b4fa5bf19497ca40154c5db2f85   

                                           To     Value  isError  
0  0x002bf459dc58584d58886169ea0e80f3ca95ffaf  0.586269        0  
1  0x6ab5cb8

In [3]:
df['TimeStamp'] = pd.to_datetime(df['TimeStamp'], unit='s')


starting feature engineering

In [4]:
import pandas as pd

# Load dataset
df = pd.read_csv("/kaggle/input/ethereum-transactions-for-fraud-detection/second_order_df.csv")
df['TimeStamp'] = pd.to_datetime(df['TimeStamp'], unit='s')

# -------------------------
# 1. Basic sent & received
# -------------------------
sent = df.groupby("From").agg(
    sent_txn_count=("TxHash", "count"),
    sent_value_total=("Value", "sum"),
    sent_value_avg=("Value", "mean"),
    sent_value_std=("Value", "std"),
    sent_error_rate=("isError", "mean"),  # % of failed txns
    first_sent=("TimeStamp", "min"),
    last_sent=("TimeStamp", "max"),
).reset_index().rename(columns={"From": "Address"})

received = df.groupby("To").agg(
    recv_txn_count=("TxHash", "count"),
    recv_value_total=("Value", "sum"),
    recv_value_avg=("Value", "mean"),
    recv_value_std=("Value", "std"),
    first_recv=("TimeStamp", "min"),
    last_recv=("TimeStamp", "max"),
).reset_index().rename(columns={"To": "Address"})

# Merge sender + receiver features
wallets = pd.merge(sent, received, on="Address", how="outer")

# Replace missing values carefully:
# - numeric cols -> 0
# - datetime cols -> NaT
for col in wallets.columns:
    if "first" in col or "last" in col:
        wallets[col] = pd.to_datetime(wallets[col], errors="coerce")
    else:
        wallets[col] = wallets[col].fillna(0)

# Active time span
wallets["first_txn"] = wallets[["first_sent", "first_recv"]].min(axis=1)
wallets["last_txn"] = wallets[["last_sent", "last_recv"]].max(axis=1)
wallets["account_age_days"] = (wallets["last_txn"] - wallets["first_txn"]).dt.days.fillna(0)

# Txn frequency
wallets["txn_count_total"] = wallets["sent_txn_count"] + wallets["recv_txn_count"]
wallets["txn_per_day"] = wallets["txn_count_total"] / (wallets["account_age_days"] + 1)

# Reliability
wallets["success_rate"] = 1 - wallets["sent_error_rate"].fillna(0)

print(wallets.head())



                                      Address  sent_txn_count  \
0  0x0000000000000000000000000000000000000000             0.0   
1  0x0000000000000000000000000000000000000001             0.0   
2  0x0000000000000000000000000000000000000003             0.0   
3  0x0000000000000000000000000000000000000004             0.0   
4  0x0000000000000000000000000000000000000005             0.0   

   sent_value_total  sent_value_avg  sent_value_std  sent_error_rate  \
0               0.0             0.0             0.0              0.0   
1               0.0             0.0             0.0              0.0   
2               0.0             0.0             0.0              0.0   
3               0.0             0.0             0.0              0.0   
4               0.0             0.0             0.0              0.0   

  first_sent last_sent  recv_txn_count  recv_value_total  recv_value_avg  \
0        NaT       NaT          2675.0      9.547368e+03    3.569109e+00   
1        NaT       NaT  

In [5]:
creating ratios required to understand the data nd make decisions

SyntaxError: invalid syntax (608451914.py, line 1)

In [ ]:
# Ratios
wallets["sent_recv_ratio"] = wallets["sent_txn_count"] / (wallets["recv_txn_count"] + 1)
wallets["value_sent_recv_ratio"] = wallets["sent_value_total"] / (wallets["recv_value_total"] + 1)

# Average transaction value (across all txns)
wallets["avg_txn_value"] = (wallets["sent_value_total"] + wallets["recv_value_total"]) / (wallets["txn_count_total"] + 1)

# Transaction consistency: variance in daily activity
wallets["txn_value_variance"] = wallets[["sent_value_std", "recv_value_std"]].mean(axis=1)

# Activity intensity
wallets["activity_rate"] = wallets["txn_count_total"] / (wallets["account_age_days"] + 1)

# Reliability (success rate already there)
wallets["fail_rate"] = 1 - wallets["success_rate"]


In [ ]:
# Ratio features (behavioral)
wallets["txn_sent_recv_ratio"] = (wallets["sent_txn_count"] + 1) / (wallets["recv_txn_count"] + 1)
wallets["value_sent_recv_ratio"] = (wallets["sent_value_total"] + 1) / (wallets["recv_value_total"] + 1)
wallets["avg_value_sent_recv_ratio"] = (wallets["sent_value_avg"] + 1e-6) / (wallets["recv_value_avg"] + 1e-6)

# Replace inf with nan and then fill
wallets.replace([float("inf"), -float("inf")], np.nan, inplace=True)
wallets.fillna(0, inplace=True)

# Fill NaNs only for numeric columns
num_cols = wallets.select_dtypes(include=[np.number]).columns
wallets[num_cols] = wallets[num_cols].fillna(0)


# Quick sanity check
print(wallets[["txn_sent_recv_ratio", "value_sent_recv_ratio", "avg_value_sent_recv_ratio"]].head())


scaling the dataset

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Drop non-numeric columns (addresses, timestamps, hashes)
numeric_cols = wallets.select_dtypes(include=[np.number]).columns
X = wallets[numeric_cols]

# 2. Initialize the scaler
scaler = StandardScaler()

# 3. Fit and transform the numeric data
X_scaled = scaler.fit_transform(X)

# 4. Convert back to DataFrame for convenience
X_scaled = pd.DataFrame(X_scaled, columns=numeric_cols, index=wallets.index)

# 5. (Optional) save the scaler for later use in inference
import joblib
joblib.dump(scaler, "scaler.pkl")

print(X_scaled.head())


defining the autoencoder model

In [ ]:
from tensorflow.keras import layers, models

input_dim = X_scaled.shape[1]  # number of features in your dataset

# Encoder
input_layer = layers.Input(shape=(input_dim,))
encoder = layers.Dense(64, activation="relu")(input_layer)
encoder = layers.Dense(32, activation="relu")(encoder)
encoder = layers.Dense(16, activation="relu")(encoder)

# Bottleneck (compressed representation)
bottleneck = layers.Dense(8, activation="relu")(encoder)

# Decoder
decoder = layers.Dense(16, activation="relu")(bottleneck)
decoder = layers.Dense(32, activation="relu")(decoder)
decoder = layers.Dense(64, activation="relu")(decoder)
output_layer = layers.Dense(input_dim, activation="linear")(decoder)

# Build autoencoder
autoencoder = models.Model(inputs=input_layer, outputs=output_layer)

# Compile
autoencoder.compile(optimizer="adam", loss="mse")

autoencoder.summary()


training the autoencoder

In [ ]:
# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)

# Train
history = autoencoder.fit(
    X_train, X_train,        # input = output for unsupervised autoencoder
    epochs=50,
    batch_size=256,
    validation_data=(X_test, X_test),
    verbose=1
)


creating proxy: calculating reconstruction error for each user

In [ ]:
# Predict reconstruction
X_pred = autoencoder.predict(X_scaled)

# Compute mean squared error per sample
reconstruction_error = np.mean(np.square(X_scaled - X_pred), axis=1)

# Add to DataFrame
wallets['reconstruction_error'] = reconstruction_error

# Quick check
wallets[['reconstruction_error']].describe()


In [ ]:
# Let's first look at the reconstruction errors
errors = wallets['reconstruction_error']

# Min-max scaling: lower error → higher score
wallets['credit_score'] = 100 * (1 - (errors - errors.min()) / (errors.max() - errors.min()))

# Optional: clip to 0–100 just in case of rounding issues
wallets['credit_score'] = wallets['credit_score'].clip(0, 100)

# Quick look
wallets[['Address', 'reconstruction_error', 'credit_score']].head(10)


saving the keras model

In [ ]:
# Save in native Keras format (.keras) — recommended
autoencoder.save("wallet_credit_autoencoder.keras")
print("Model saved successfully ✅")



loading the mdel

In [ ]:
from tensorflow.keras.models import load_model

# Load the saved model
autoencoder = load_model("wallet_credit_autoencoder.keras")
print("Model loaded successfully ✅")


In [ ]:
autoencoder.save("/kaggle/working/wallet_credit_autoencoder.keras")
